In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('data.sqlite')

In [2]:
q = """
SELECT lastName, firstName, officeCode
FROM employees
JOIN offices
USING(officeCode)
WHERE country = "USA"
;
"""
pd.read_sql(q, conn)

,lastName,firstName,officeCode
0,Bow,Anthony,1
1,Firrelli,Jeff,1
2,Jennings,Leslie,1
3,Murphy,Diane,1
4,Patterson,Mary,1
5,Thompson,Leslie,1
6,Firrelli,Julie,2
7,Patterson,Steve,2
8,Tseng,Foon Yue,3
9,Vanauf,George,3


### Using A SubQuery for the Join above

In [3]:
q = """
SELECT firstName, lastName, officeCode
FROM employees
WHERE officeCode IN (
SELECT officeCode
FROM offices
WHERE country = "USA"
);
"""
pd.read_sql(q, conn)

,firstName,lastName,officeCode
0,Diane,Murphy,1
1,Mary,Patterson,1
2,Jeff,Firrelli,1
3,Anthony,Bow,1
4,Leslie,Jennings,1
5,Leslie,Thompson,1
6,Julie,Firrelli,2
7,Steve,Patterson,2
8,Foon Yue,Tseng,3
9,George,Vanauf,3


In [4]:
q = """
SELECT firstName, lastName, officeCode
FROM employees
WHERE officeCode IN(
SELECT officeCode
FROM offices
JOIN employees
USING(officeCode)
GROUP BY 1
HAVING COUNT(employeeNumber) >=5
);
"""

pd.read_sql(q, conn)

,firstName,lastName,officeCode
0,Diane,Murphy,1
1,Mary,Patterson,1
2,Jeff,Firrelli,1
3,Gerard,Bondur,4
4,Anthony,Bow,1
5,Leslie,Jennings,1
6,Leslie,Thompson,1
7,Loui,Bondur,4
8,Gerard,Hernandez,4
9,Pamela,Castillo,4


### Chaining Aggregates

In [5]:
q = """
SELECT AVG(customerAvgPayment) AS averagePayment
FROM (
    SELECT AVG(amount) AS customerAvgPayment
    FROM payments
    JOIN customers
        USING(customerNumber)
    GROUP BY customerNumber
)
;"""
pd.read_sql(q, conn)

,averagePayment
0,31489.754582


In [6]:
q = """
SELECT lastName, firstName, employeeNumber
FROM employees
WHERE employeeNumber IN (SELECT salesRepEmployeeNumber
                     FROM customers 
                     WHERE country = "USA")
;
"""
pd.read_sql(q, conn)

,lastName,firstName,employeeNumber
0,Jennings,Leslie,1165
1,Thompson,Leslie,1166
2,Firrelli,Julie,1188
3,Patterson,Steve,1216
4,Tseng,Foon Yue,1286
5,Vanauf,George,1323


In [7]:
conn.close()